# Cleaning data
In dit document gaan we alle features schoonmaken en een nieuwe dataset aanmaken waar all die features opgeschoont zijn.   
Dit zijn de features die we gaan opschonen: 
stm_sap_meld_ddt: Tijd van melding, continu.    
stm_geo_mld: Geolocatie, nominaal.    
stm_equipm_nr_mld: Nummer van equipement, nominaal.   
stm_equipm_soort_mld: Soort van equipement, nominaal.   
stm_prioriteit: Prioriteitsindex, nominaal.    
stm_aanngeb_dd: Datum aannemer gebeld, continu.    
stm_oorz_groep: Oorzaak groep storing, nominaal.   
stm_oorz_code: Oorzaak code storing, nominaal.    
stm_contractgeb_gst: Contractgebied aannnemer, nominaal.   
stm_techn_gst: Techniekveld melding, nominaal.   
stm_progfh_in_duur: Prognose aannemer duur functiehersteltijd, continu.    
stm_fh_tijd'
    ]

In [16]:
import numpy as np
import pandas as pd

In [17]:
file = pd.read_csv("sap_storing_data_hu_project.csv", low_memory=False)

## Duplicaten verwijderen en dataframe aanmaken
Eerst kijken we of er duplicates in de dataset zitten

In [217]:
file["#stm_sap_meldnr"].duplicated().sum()

0

Hier zien we dat er inderdaad duplicaten in de dataset zitten, deze gaan we verwijderen

In [218]:
file = file.drop_duplicates(subset=["#stm_sap_meldnr"])

Nu maken we een dataframe aan met alle kolommen die we gaan schoonmaken

In [251]:
df = file[['stm_sap_meld_ddt', 'stm_geo_mld', 'stm_equipm_soort_mld', 'stm_equipm_nr_mld', 'stm_prioriteit', 'stm_aanngeb_dd', 'stm_oorz_groep', 'stm_oorz_code', 'stm_contractgeb_gst', 'stm_techn_gst', 'stm_progfh_in_duur']].copy()
df.columns

Index(['stm_sap_meld_ddt', 'stm_geo_mld', 'stm_equipm_soort_mld',
       'stm_equipm_nr_mld', 'stm_prioriteit', 'stm_aanngeb_dd',
       'stm_oorz_groep', 'stm_oorz_code', 'stm_contractgeb_gst',
       'stm_techn_gst', 'stm_progfh_in_duur'],
      dtype='object')

Nu gaan we de kolommen in deze dataframe 1 voor 1 schoonmaken.

## stm_sap_meld_ddt
Eerst verwijderen we de NaN

In [252]:
df = df.dropna(subset=['stm_sap_meld_ddt'])
df['stm_sap_meld_ddt']

1         02/01/2006 09:00:00
2         02/01/2006 12:35:00
3         02/01/2006 16:40:00
4         02/01/2006 22:30:00
5         02/01/2006 11:23:00
                 ...         
898516    11/05/2013 07:55:00
898518    11/05/2013 07:59:00
898520    11/05/2013 08:06:00
898522    11/05/2013 09:21:00
898524    20/08/2016 14:15:17
Name: stm_sap_meld_ddt, Length: 566480, dtype: object

In deze kolom staan datums samen met de tijden van van de dag. Hier kunnen we heel makkelijk verkeerde values eruit halen met to_datetime en errors='coerce'. Dit zet namelijk alle verkeerde tijden om naar een NaN, deze kunnen we daarna heel makelijk verwijderen

In [253]:
df['stm_sap_meld_ddt'] = pd.to_datetime(df['stm_sap_meld_ddt'], errors='coerce')

df = df.dropna(subset=['stm_sap_meld_ddt'])
df['stm_sap_meld_ddt']

1        2006-02-01 09:00:00
2        2006-02-01 12:35:00
3        2006-02-01 16:40:00
4        2006-02-01 22:30:00
5        2006-02-01 11:23:00
                 ...        
898510   2013-11-05 07:17:00
898516   2013-11-05 07:55:00
898518   2013-11-05 07:59:00
898520   2013-11-05 08:06:00
898522   2013-11-05 09:21:00
Name: stm_sap_meld_ddt, Length: 225165, dtype: datetime64[ns]

Nu hebben we alle verkeerde values eruit gehaald, dit waren er zo te zien een hoop. Nu moeten we de tijden nog omzetten naar een getal die in model in kan. Dit doen met met de Unix-tijdstempel (1970-01-01 00:00:00). 

In [254]:
df['stm_sap_meld_ddt'] = df['stm_sap_meld_ddt'].astype('int64')
df['stm_sap_meld_ddt']

1         1138784400000000000
2         1138797300000000000
3         1138812000000000000
4         1138833000000000000
5         1138792980000000000
                 ...         
898510    1383635820000000000
898516    1383638100000000000
898518    1383638340000000000
898520    1383638760000000000
898522    1383643260000000000
Name: stm_sap_meld_ddt, Length: 225165, dtype: int64

Nu hebben we er int van gemaakt die het aantal seconden sinds 1970-01-01 00:00:00 weergeeft

## stm_geo_mld
Eerst verwijderen we de NaN

In [255]:
df = df.dropna(subset=['stm_geo_mld'])
df['stm_geo_mld']

1         624.0
2         201.0
3          25.0
4          12.0
5         614.0
          ...  
898510    155.0
898516    118.0
898518    158.0
898520    560.0
898522    468.0
Name: stm_geo_mld, Length: 222317, dtype: object

Eerst kijken we naar alle unieke waardes.

In [256]:
print(sorted(set(df["stm_geo_mld"])))

['0', '001', '002', '004', '005', '006', '007', '008', '009', '011', '012', '013', '015', '017', '018', '019', '020', '021', '023', '024', '025', '026', '027', '028', '030', '031', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '046', '047', '049', '050', '051', '052', '053', '054', '055', '056', '057', '058', '059', '060', '062', '063', '064', '065', '067', '070', '071', '073', '074', '075', '076', '078', '079', '080', '082', '083', '084', '085', '087', '088', '089', '090', '091', '092', '093', '094', '095', '096', '097', '098', '099', '1', '1.0', '10', '10.0', '100', '100.0', '101', '101.0', '102', '102.0', '103', '103.0', '104', '104.0', '105', '105.0', '106', '106.0', '107', '107.0', '108', '108.0', '109', '109.0', '11', '11.0', '110', '110.0', '111', '111.0', '112', '112.0', '114', '114.0', '115', '115.0', '116', '116.0', '117', '117.0', '118', '118.0', '119', '119.0', '12', '12.0', '120', '120.0', '121', '121.0', '122', '122.0', '123', '123.0'

Zo te zien staan er int en floats in de dataset als geocodes, deze horen hetzelfde te zijn, dus deze gaan we hetzelfde maken.

In [257]:
df['stm_geo_mld'] = df['stm_geo_mld'].astype(float).astype(int)
print(sorted(set(df["stm_geo_mld"])))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 143, 144, 145, 146, 149, 152, 155, 158, 161, 163, 164, 165, 166, 200, 201, 203, 204, 205, 206, 208, 209, 210, 211, 212, 213, 216, 217, 223, 226, 228, 305, 309, 400, 405, 428, 429, 432, 433, 435, 437, 438, 451, 452, 466, 467, 468, 469, 470, 474, 475, 476, 477, 478, 479, 485, 486, 490, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 52

Nu zijn alle geocodes int, dus is het nu opgeschoont.

# stm_equipm_soort_mld
Eerst verwijderen we de NaN

In [258]:
df = df.dropna(subset=['stm_equipm_soort_mld'])
df['stm_equipm_soort_mld']

5         DETECTREIN
8            OVERWEG
22           OVERWEG
47        BAANLICHAA
63        BAANLICHAA
             ...    
898508     DETGRSSSL
898510        WISSEL
898516    TREINBEINV
898518    GELUIDSSCH
898522       OVERWEG
Name: stm_equipm_soort_mld, Length: 163171, dtype: object

In [259]:
print(dict(df["stm_equipm_soort_mld"].value_counts()))

{'WISSEL': 23595, 'OVERWEG': 16757, 'SEIN': 13339, 'TBHHARDW': 9751, 'DETGRSSSL': 6816, 'PLAATSBEP': 6545, 'ICTINFRA': 6308, 'VISUELERIS': 6080, 'DETECTREIN': 4935, 'RISANALOOG': 3811, 'TREINBEINV': 2838, 'TIJDKLOK': 2814, 'TUNCONSTR': 2422, 'SPOORBRUG': 2305, 'HEKWERK': 2272, 'OMROEPOP': 2227, 'BAANLICHAA': 2196, 'SPOORTAK': 2089, 'SPOORTUN': 1726, 'TANKINST': 1550, 'SPOORSPS': 1314, 'SNSCHAKAUT': 1257, 'OMROEPHP': 1174, 'GEBOUW': 1147, 'TTI': 1074, 'SEIN_BORD': 963, 'LVNOODSTGR': 956, 'BVLSCHAK': 917, 'AARDFTDET': 892, 'INTLOCKUNT': 850, 'WISVERWINS': 823, 'GSMR': 806, 'WISSELSTEL': 787, 'WERKPLBEV': 746, 'LOC_VOED': 709, 'TELEFOONS': 642, 'PSV': 622, 'PERRON': 608, 'WISSCONS': 540, 'BOVENLEID': 539, 'OVERWBEVL': 533, 'DETPSSSL': 528, 'SYSTBOVENL': 524, '3KVOMVCENT': 511, 'TERREINVL': 490, 'DETJADE1SL': 484, 'STATOMROEP': 475, 'NUTSWATER': 475, 'CABOP': 475, 'HANGDRAAD': 468, 'INTERLOCK': 465, 'BALLAST': 463, 'BEV_KABEL': 446, 'TELCENTRAL': 419, 'SPOORDRKW': 411, 'SPOORDWL': 390, 'PL

We zien geen spelfouten of andere datatypes, dus kunnen vastellen dat deze kolom geen verkeerde data bevat.

## stm_prioriteit
Eerst verwijderen we de NaN

In [260]:
df = df.dropna(subset=['stm_prioriteit'])
df['stm_prioriteit']

5         9.0
8         9.0
22        9.0
47        9.0
63        9.0
         ... 
898508    2.0
898510    4.0
898516    4.0
898518    5.0
898522    2.0
Name: stm_prioriteit, Length: 163170, dtype: float64

In [261]:
print(dict(df["stm_prioriteit"].value_counts()))

{5.0: 54016, 2.0: 51997, 4.0: 46410, 9.0: 8905, 8.0: 1209, 1.0: 633}


Hier zien we dat volgens het meegegeven document, in de kolom geen verkeerde data zit

Uit het meegevenen document kunnen we ook vastellen, dat elke rij met stm_prioriteit als 9.0 verwijderd moet worden, omdat hierbij geen aannemer naartoe hoeft.

In [262]:
df = df[~df["stm_prioriteit"].isin([9.0])]

## stm_aanngeb_dd
Eerst verwijderen we de NaN

In [263]:
df = df.dropna(subset=['stm_aanngeb_dd'])
df['stm_aanngeb_dd']

85        10/01/2006
351       05/02/2006
649       06/03/2006
664       07/03/2006
2148      01/07/2006
             ...    
898508    11/05/2013
898510    11/05/2013
898516    11/05/2013
898518    11/05/2013
898522    11/05/2013
Name: stm_aanngeb_dd, Length: 154109, dtype: object

Zoals we zien is dit een datetime, hier halen we dus de verkeerde tijden uit

In [264]:
df['stm_aanngeb_dd'] = pd.to_datetime(df['stm_aanngeb_dd'], errors='coerce')

df = df.dropna(subset=['stm_aanngeb_dd'])
df['stm_aanngeb_dd']

85       2006-10-01
351      2006-05-02
649      2006-06-03
664      2006-07-03
2148     2006-01-07
            ...    
898508   2013-11-05
898510   2013-11-05
898516   2013-11-05
898518   2013-11-05
898522   2013-11-05
Name: stm_aanngeb_dd, Length: 153969, dtype: datetime64[ns]

Nu hebben we alle verkeerde values eruit gehaald. Nu moeten we de tijden nog omzetten naar een getal die in model in kan. Dit doen met met de Unix-tijdstempel (1970-01-01). 

In [265]:
df['stm_aanngeb_dd'] = df['stm_aanngeb_dd'].astype('int64')
df['stm_aanngeb_dd']

85        1159660800000000000
351       1146528000000000000
649       1149292800000000000
664       1151884800000000000
2148      1136592000000000000
                 ...         
898508    1383609600000000000
898510    1383609600000000000
898516    1383609600000000000
898518    1383609600000000000
898522    1383609600000000000
Name: stm_aanngeb_dd, Length: 153969, dtype: int64

Nu hebben we er int van gemaakt die het aantal seconden sinds 1970-01-01 weergeeft

## stm_oorz_groep
Eerst verwijderen we de NaN

In [266]:
df = df.dropna(subset=['stm_oorz_groep'])
df['stm_oorz_groep']

85        ONR-DERD
351       ONR-DERD
649        ONR-RIB
664       ONR-DERD
2148       TECHONV
            ...   
898508     TECHONV
898510     TECHONV
898516     TECHONV
898518     TECHONV
898522     TECHONV
Name: stm_oorz_groep, Length: 143878, dtype: object

In [267]:
print(dict(df["stm_oorz_groep"].value_counts()))

{'TECHONV': 100915, 'ONR-DERD': 24204, 'ONR-RIB': 13298, 'WEER': 5461}


Hier zien we dat er geen verkeerde waarde in staan omdat er geen spelfouten of rare waarde in staan, dus is deze kolom te vertrouwen

## stm_oorz_code
Eerst verwijderen we de NaN

In [268]:
df = df.dropna(subset=['stm_oorz_code'])
df['stm_oorz_code']

85        147.0
351       145.0
649       133.0
664       151.0
2148      218.0
          ...  
898508    215.0
898510    298.0
898516    221.0
898518    298.0
898522    218.0
Name: stm_oorz_code, Length: 143878, dtype: float64

In [269]:
print(dict(df["stm_oorz_code"].value_counts()))

{221.0: 23723, 218.0: 19323, 215.0: 16653, 294.0: 9171, 135.0: 7000, 145.0: 6578, 151.0: 6123, 298.0: 5858, 133.0: 5808, 213.0: 4194, 225.0: 3215, 147.0: 2870, 140.0: 2459, 212.0: 2123, 203.0: 2071, 143.0: 1600, 219.0: 1416, 230.0: 1332, 241.0: 1186, 227.0: 1145, 146.0: 1138, 207.0: 1120, 181.0: 1089, 183.0: 1076, 226.0: 1065, 186.0: 998, 150.0: 937, 149.0: 873, 299.0: 871, 182.0: 787, 209.0: 763, 228.0: 690, 184.0: 653, 154.0: 607, 214.0: 591, 223.0: 524, 220.0: 522, 187.0: 497, 222.0: 471, 229.0: 451, 211.0: 436, 144.0: 400, 132.0: 379, 210.0: 353, 148.0: 315, 201.0: 311, 240.0: 308, 242.0: 279, 185.0: 189, 204.0: 166, 142.0: 161, 188.0: 147, 141.0: 142, 224.0: 123, 208.0: 111, 234.0: 96, 250.0: 56, 235.0: 50, 131.0: 49, 231.0: 38, 134.0: 37, 999.0: 34, 239.0: 31, 189.0: 23, 206.0: 20, 130.0: 16, 202.0: 11, 233.0: 9, 136.0: 7, 205.0: 5, 180.0: 2, 51.0: 1, 33.0: 1, 139.0: 1}


Uit het meegegeven document kunnen we vastellen dat de waarde 999 wel in de dataset staan, maar niet in het document. Deze gaan we eerst onderzoeken.

In [270]:
df["stm_oorz_code"].value_counts().loc[999.0]

34

999 komt 13 keer voor, dit is de een duidelijke fout omdat 999 een typsiche waarde is die is ingevult als de stm_oorz_code niet duidelijk is. Deze gaan we nu verwijderen.

In [271]:
df = df[~df["stm_oorz_code"].isin([999.0])]

## stm_contractgeb_gst
Eerst verwijderen we de NaN

In [272]:
df = df.dropna(subset=['stm_contractgeb_gst'])
df['stm_contractgeb_gst']

85        24.0
351       12.0
649       10.0
664       27.0
2148      29.0
          ... 
898508     1.0
898510    71.0
898516     5.0
898518    71.0
898522     4.0
Name: stm_contractgeb_gst, Length: 143844, dtype: float64

In [273]:
print(dict(df["stm_contractgeb_gst"].value_counts()))

{5.0: 6379, 9.0: 6244, 4.0: 6218, 12.0: 5335, 8.0: 4198, 26.0: 4193, 30.0: 4188, 3.0: 4171, 2.0: 4157, 11.0: 3754, 71.0: 3676, 31.0: 3641, 53.0: 3592, 24.0: 3473, 7.0: 3403, 10.0: 3392, 19.0: 3361, 32.0: 3343, 34.0: 3339, 13.0: 3330, 27.0: 3119, 21.0: 3072, 25.0: 2927, 20.0: 2871, 18.0: 2867, 62.0: 2716, 51.0: 2478, 6.0: 2469, 81.0: 2467, 37.0: 2462, 23.0: 2393, 29.0: 2386, 36.0: 2343, 52.0: 2153, 35.0: 2079, 61.0: 2023, 28.0: 1922, 1.0: 1869, 22.0: 1680, 14.0: 1604, 58.0: 1553, 63.0: 1457, 54.0: 1429, 59.0: 1349, 64.0: 1113, 55.0: 1096, 15.0: 883, 60.0: 847, 16.0: 690, 33.0: 648, 56.0: 613, 17.0: 562, 57.0: 96, 83.0: 75, 50.0: 62, 99.0: 56, 82.0: 25, 70.0: 3}


Hier zien we dat er geen verkeerde waarde in staan omdat er geen spelfouten of rare waarde in staan, dus is deze kolom te vertrouwen

## stm_techn_gst
Eerst verwijderen we de NaN

In [274]:
df = df.dropna(subset=['stm_techn_gst'])
df['stm_techn_gst']

85        X
351       S
649       S
664       B
2148      S
         ..
898508    S
898510    S
898516    S
898518    B
898522    S
Name: stm_techn_gst, Length: 143844, dtype: object

In [275]:
print(dict(df["stm_techn_gst"].value_counts()))

{'S': 55535, 'B': 28732, 'P': 18020, 'T': 17195, 'E': 11009, 'K': 8330, 'O': 3593, 'G': 584, 'M': 529, 'I': 164, 'X': 128, 'A': 25}


Hier zien we dat er geen verkeerde waarde in staan omdat er geen spelfouten of rare waarde in staan, dus is deze kolom te vertrouwen

## stm_progfh_in_duur
Eerst verwijderen we de NaN

In [276]:
df = df.dropna(subset=['stm_progfh_in_duur'])
df['stm_progfh_in_duur']

85        99999999.0
351       99999999.0
649       99999999.0
664       99999999.0
2148      99999999.0
             ...    
898508             4
898510            90
898516           180
898518             4
898522            52
Name: stm_progfh_in_duur, Length: 143844, dtype: object

De waardes in deze kolom staan in minuten, nu gaan we alle tijden eruit halen die langer zijn dan 8 uur. Dit doen we omdat tijden langer dan 8 uur onnodig zijn om te voorspellen omdat deze tijden onnodig zijn voor reizigers als wij informatie willen leveren voor hoelang het nog duurt todat te treinen weer rijden. Voor dezelfde reden halen we onder de 5 minuten er ook uit. Dit is te kort om als storing gezien te worden.

Ook zien we dat de kolom string als datatype heeft, dit moeten we omzetten naar een numeric datatype zodat we er mee kunnen rekenen.

In [277]:
# Alle waardes in de kolom omzetten naar float
df["stm_progfh_in_duur"] = pd.to_numeric(df["stm_progfh_in_duur"], errors="coerce")

df = df[df["stm_progfh_in_duur"] <= (8 * 60)] # 8 uur
df = df[df["stm_progfh_in_duur"] >= 5] # 5 minuten

# Target aanmaken
Wat wij willen voorspellen is hoelang het duurt voor de treinen weer rijden, vanaf dat de aannemer zijn prognose heeft geleverd. Om dit te berekenen gebruiken we deze features: stm_progfh_in_datum, stm_progfh_in_tijd, stm_fh_dd, stm_fh_tijd. Hierbij is:     

stm_progfh_in_invoer_dat: datum van wanneer de prognose is aangemaakt.      
stm_progfh_in_invoer_tijd: tijd van wanneer de prognose is aangemaakt.     
stm_fh_dd: datum van wanneer de treinen weer rijden.    
stm_fh_tijd: tijd van wanneer de treinen weer rijden.    

In [278]:
df1 = file[["stm_progfh_in_invoer_dat", "stm_progfh_in_invoer_tijd", "stm_fh_dd", "stm_fh_tijd"]].dropna()

# Hier zetten we alle datums om naar een datetime en halen we de verkeerde datums eruit met errors='coerce'
df1["stm_progfh_in_invoer_dat"] = pd.to_datetime(df1["stm_progfh_in_invoer_dat"], format="%d/%m/%Y", errors='coerce')
df1["stm_fh_dd"] = pd.to_datetime(df1["stm_fh_dd"], format="%d/%m/%Y", errors='coerce')
# Dit is het verschil in datums omgezet van dagen naar minuten
datum_verschil = ((df1["stm_fh_dd"] - df1["stm_progfh_in_invoer_dat"]).dt.days) * 24 * 60

# Hier zetten we alle tijden om naar een datetime en halen we de verkeerde tijden eruit met errors='coerce'
df1["stm_progfh_in_invoer_tijd"] = pd.to_datetime(df1["stm_progfh_in_invoer_tijd"], format="%H:%M:%S", errors='coerce')
df1["stm_fh_tijd"] = pd.to_datetime(df1["stm_fh_tijd"], format="%H:%M:%S", errors='coerce')
# Dit is het verschil in tijden omgezet van seconden naar minuten
tijden_verschil = (df1["stm_fh_tijd"] - df1["stm_progfh_in_invoer_tijd"]).dt.total_seconds() / 60

# Dit is target, met round() and .astype() maken we er een .0 getal van. Hierdoor 
df['stm_progfh_t_fh'] = round(datum_verschil + tijden_verschil, 0).astype(float)

# Verwijder alle foute tijden die naar NaN zijn omgezet met errors='coerce'
df = df.dropna(subset=["stm_progfh_t_fh"])

Voor dezelfde redenen als bij stm_progfh_in_duur, gaan we alles boven de 8 uur en onder de 5 minuten verwijderen. Dit verwijderd ook alle rijen waar de treinen eerder weer reden dan de prognose was opgeleverd. Dit waren dus niet te vertrouwen rijen.

In [279]:
df = df[df["stm_progfh_t_fh"] <= (8 * 60)] # 8 uur
df = df[df["stm_progfh_t_fh"] >= 5] # 5 minuten


Als laatst sorteren we nog op de meldtijd, hierdoor bestaat onze testset uit de meest recentelijke data, dit geeft dus de meest realistische score voor als we in het echt van nieuwe data de target willen voorspellen. Sorteren doen we op de datum en tijd van wanneer de melding is gemeld.

In [280]:
df = df.sort_values(by='stm_sap_meld_ddt', ascending=True)
# Index resetten voor de netheid.
df.reset_index()

,index,stm_sap_meld_ddt,stm_geo_mld,stm_equipm_soort_mld,stm_equipm_nr_mld,stm_prioriteit,stm_aanngeb_dd,stm_oorz_groep,stm_oorz_code,stm_contractgeb_gst,stm_techn_gst,stm_progfh_in_duur,stm_progfh_t_fh
0,215649,1136091231000000000,200,OVERWEG,10211104.0,2.0,1136073600000000000,ONR-DERD,145.0,37.0,S,60.0,10.0
1,215642,1136091975000000000,91,OVERWEG,10247725.0,5.0,1136073600000000000,ONR-DERD,145.0,12.0,S,36.0,13.0
2,215654,1136103176000000000,212,OVERWEG,10211398.0,2.0,1136073600000000000,ONR-DERD,145.0,30.0,S,139.0,43.0
3,215657,1136106750000000000,608,OVERWEG,10211489.0,2.0,1136073600000000000,ONR-DERD,145.0,31.0,S,102.0,81.0
4,215663,1136112094000000000,536,RISANALOOG,10002607.0,5.0,1136073600000000000,TECHONV,212.0,2.0,T,45.0,157.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23133,894727,1575632104000000000,133,BATTERIJ,11611925.0,2.0,1575590400000000000,TECHONV,215.0,8.0,E,232.0,129.0
23134,894742,1575633684000000000,12,PLAATSBEP,11563592.0,5.0,1575590400000000000,ONR-DERD,143.0,35.0,O,89.0,10.0
23135,214157,1575640838000000000,79,SPOORBRUG,10329041.0,5.0,1575590400000000000,TECHONV,225.0,6.0,K,219.0,6.0
23136,214156,1575640838000000000,79,SPOORBRUG,10332822.0,5.0,1575590400000000000,TECHONV,230.0,6.0,K,171.0,24.0


Nu is alle data opgeschoont en is de target aangemaakt. Nu kunnen we alles naar een csv schrijven zodat we dit in een ander notebook kunnen gebruiken.

In [281]:
df.to_csv('cleaned_data.csv', index=False)